# Paper 1 — Interactive analysis notebook

Reproduces every table and figure in **Paper 1: TrimTree**, plus gives hooks
for your own hypotheses. All data loads from `../data/` — no cloud calls.

**Structure**:
1. Setup + data loading
2. Table A — Algorithmic p₁ × strategy × budget (E1)
3. Table B — LLM accuracy × model × strategy × budget (E2)
4. Table C — Cost efficiency
5. Hypothesis checks (H1…H5) — with adjustable thresholds
6. Per-repo breakdown (anti-Django-bias check)
7. n_candidates bucket analysis
8. Correlation analysis (H3 reconciliation)
9. **Your own hypotheses** — ready-to-extend cells

## 1. Setup + data loading

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

DATA = Path('../data').resolve()
assert DATA.exists(), f'Expected {DATA} — run notebook from docs/research/notebooks/'
print('Data directory:', DATA)
print('Files:', sorted(p.name for p in DATA.iterdir() if p.is_file())[:20])

In [ ]:
# Load all aggregates into a namespace
swe_gold = pq.read_table(DATA / 'swe_bench_gold.parquet').to_pandas()
swe_strat = pq.read_table(DATA / 'swe_bench_strategy_results.parquet').to_pandas()

# Per-model raw LLM results
llm_files = sorted((DATA / 'llm_results').glob('*.parquet'))
llm = pd.concat([pq.read_table(f).to_pandas() for f in llm_files], ignore_index=True)

# CSVs
table_a = pd.read_csv(DATA / 'swe_bench_strategies.csv')
table_b = pd.read_csv(DATA / 'swe_bench_llm.csv')
table_c = pd.read_csv(DATA / 'swe_bench_cost.csv')

print(f'SWE-bench tasks:           {len(swe_gold)}')
print(f'Strategy-results rows:     {len(swe_strat)}')
print(f'LLM results rows:          {len(llm)}')
print(f'Unique models in LLM data: {sorted(llm.model.unique())}')
print(f'Strategies:                {sorted(llm.strategy.unique())}')

## 2. Table A — algorithmic p₁ × strategy × budget

In [ ]:
piv_a = (swe_strat.groupby(['strategy','budget_tok']).p1.mean()
         .unstack().round(3))
piv_a

## 3. Table B — LLM accuracy × model × strategy × budget

In [ ]:
piv_b = (llm.groupby(['model','strategy','budget_tok']).is_correct.mean()
         .unstack().round(3))
piv_b

## 4. Table C — cost efficiency

Sorted by `$/correct`. Lower is better.

In [ ]:
table_c.sort_values('cost_per_correct_usd')

## 5. Hypothesis checks — adjustable thresholds

Change the thresholds to see the verdict shift.

In [ ]:
def check_h1(df, budget=2000, threshold=0.10):
    """Priority-KW > FIFO on algo p1. Original threshold = +0.10."""
    fifo = df.query('strategy=="fifo" and budget_tok==@budget').p1.mean()
    kw = df.query('strategy=="priority_kw" and budget_tok==@budget').p1.mean()
    delta = kw - fifo
    return dict(delta=delta, fifo=fifo, kw=kw,
                verdict='PASS' if delta >= threshold else 'FAIL')

print('H1:', check_h1(swe_strat))

In [ ]:
def check_h5(df, threshold_pp=10.0):
    """Local-tier ≈ frontier-tier when gold is in compressed set."""
    sub = df[df.gold_in_selected].copy()
    acc = sub.groupby('model').is_correct.mean()
    frontier = [m for m in acc.index if 'claude' in m.lower()]
    local = [m for m in acc.index if any(k in m for k in ('gemma', 'gpt-oss', 'qwen'))]
    if not frontier or not local:
        return dict(verdict='SKIP', reason='need both tiers')
    gap_pp = (acc[frontier].max() - acc[local].max()) * 100
    return dict(frontier_max=float(acc[frontier].max()),
                local_max=float(acc[local].max()),
                gap_pp=gap_pp,
                verdict='PASS' if gap_pp <= threshold_pp else 'FAIL')

print('H5:', check_h5(llm))

## 6. Per-repo breakdown (anti-Django-bias check)

Django = 46% of SWE-bench Verified. Reviewer will ask: *does Priority work on other repos too?*

In [ ]:
# Join strategy results with repo info
repo_map = dict(zip(swe_gold.task_id, swe_gold.repo))
swe_strat['repo'] = swe_strat.task_id.map(repo_map)

per_repo = (swe_strat[swe_strat.strategy.isin(['fifo','priority_kw','priority_kw_fallback'])]
            .query('budget_tok==2000')
            .groupby(['repo','strategy']).p1.mean().unstack()
            .round(3))
per_repo['tasks'] = swe_strat.query('strategy=="fifo" and budget_tok==2000').groupby('repo').size()
per_repo.sort_values('tasks', ascending=False)

## 7. n_candidates bucket analysis

Where does compression actually matter? Answer: medium + large buckets.

In [ ]:
def bucket(n):
    return 'small_1-5' if n <= 5 else 'medium_6-20' if n <= 20 else 'large_21+'

swe_strat['bucket'] = swe_strat.n_candidates.apply(bucket)

piv_bucket = (swe_strat.query('budget_tok==2000 and strategy in ["fifo","priority_kw","priority_kw_fallback"]')
              .groupby(['bucket','strategy']).agg(p1_mean=('p1','mean'), n=('p1','count'))
              .round(3))
piv_bucket

## 8. Correlation analysis (H3 reconciliation)

LLM accuracy vs algorithmic p₁ — per-model Pearson r.

H3 fails (r < 0.85) for all models; the interesting finding is the sign.

In [ ]:
# Join LLM accuracy with algo p1 per (task, strategy, budget)
joined = llm.merge(swe_strat[['task_id','strategy','budget_tok','p1']],
                   on=['task_id','strategy','budget_tok'], how='inner')

rows = []
for model, g in joined.groupby('model'):
    cells = g.groupby(['strategy','budget_tok']).agg(algo=('p1','mean'), llm_acc=('is_correct','mean'))
    if len(cells) < 3:
        rows.append(dict(model=model, r=None, n_cells=len(cells)))
        continue
    r = np.corrcoef(cells['algo'], cells['llm_acc'])[0,1]
    rows.append(dict(model=model, r=r, n_cells=len(cells)))

pd.DataFrame(rows).round(3)

## 9. Your own hypotheses — scratch cells

Examples to extend:

### Hypothesis: "budget sweet spot" — where does a larger budget stop helping?

In [ ]:
# Does LLM accuracy plateau at some budget?
sweet = llm.groupby(['model','strategy','budget_tok']).is_correct.mean().reset_index()
sweet_wide = sweet.pivot_table(index=['model','strategy'], columns='budget_tok', values='is_correct').round(3)
sweet_wide['Δ(8k-1k)'] = sweet_wide[8000] - sweet_wide[1000]
sweet_wide.sort_values('Δ(8k-1k)', ascending=False)

### Hypothesis: does confidence correlate with correctness?

In [ ]:
# Well-calibrated model: high confidence → more likely correct
for model, g in llm[llm.confidence > 0].groupby('model'):
    hi = g[g.confidence >= 0.8].is_correct.mean()
    lo = g[g.confidence < 0.8].is_correct.mean()
    print(f'{model:24s}  high-conf acc={hi:.3f}  low-conf acc={lo:.3f}  '
          f'calibration_gap={hi-lo:+.3f}')

### Hypothesis: chosen_file token-overlap with issue correlates with correctness?

In [ ]:
# (Requires joining chosen_file back with issue text from swe_gold)
issue_map = dict(zip(swe_gold.task_id, swe_gold.problem_statement))
llm['issue_text'] = llm.task_id.map(issue_map)

import re
def overlap_score(path, issue):
    if not path or not issue: return 0.0
    p_tokens = set(t.lower() for t in re.split(r'[/_\-.]', path) if len(t) >= 3)
    i_tokens = set(t.lower() for t in re.split(r'\W+', issue) if len(t) >= 3)
    if not p_tokens: return 0.0
    return len(p_tokens & i_tokens) / len(p_tokens)

llm['chosen_overlap'] = llm.apply(lambda r: overlap_score(r.chosen_file, r.issue_text), axis=1)

print(llm.groupby(pd.cut(llm.chosen_overlap, [0, 0.25, 0.5, 0.75, 1.01], include_lowest=True))
      .is_correct.agg(['mean','count']).round(3))

### Now make your own:

The full joined frame is in `joined` (LLM × strategy p₁). Add cells below
and explore. Suggestions:

- Does thinking-enabled model accuracy scale with candidate list length?
- How does `n_selected` (items emitted after knapsack) correlate with accuracy?
- Are there repos where Priority-ALL (the `priority_all` strategy in E1)
  actually outperforms Priority-KW? (data in `swe_strat`, check `.query('strategy==\"priority_all\"')`)
- Per-task cache-efficiency on Sonnet/Opus: `tokens_cache_read / (tokens_cache_read + tokens_prompt_fresh)`